# Python Day 4 강의 노트북 — NumPy·pandas 입문 및 미니 프로젝트

**HYUNDAI AI Insight Campus 온디바이스 AI · 프로그래밍 언어(Python) 4/4 · 9/7(월)**

| 교시 | 주제 | 애니메이션 |
|---|---|---|
| 1 | NumPy 기초 — 벡터 연산·브로드캐스팅 | D4-1 ①~④ |
| 2 | NumPy 활용 — 마스크·통계·이미지=배열 | D4-1 ⑤~⑦ |
| 3 | Matplotlib — 시계열·히스토그램 | (실습 중심) |
| 4 | pandas 맛보기 — DataFrame·groupby·CSV | D4-2 ①~⑥ |
| 5 | 파이썬 문법 마무리 — 제너레이터·이터레이터 | D3-5 재사용 |
| 6 | 미니 프로젝트 ① 필수 | 전체 |
| 7 | 미니 프로젝트 ② 확장·시연 준비 | 전체 |
| 8 | 팀별 시연·코드 리뷰·정리 | — |

### 이 노트북 사용법 (Day 1~3과 동일)
- **▶ 예제** 셀은 그대로 실행하며 출력을 함께 읽습니다.
- **✏️ 빈칸 채우기**는 `____` 를 채워 실행합니다. 채우기 전 실행 오류는 정상입니다.
- **✏️ 괄호 넣기**는 ( ) 를 생각한 뒤 정답 셀로 확인합니다.
- **🔒 정답 보기** 셀은 단독 실행됩니다. 강사 신호 후에 여세요.

In [ ]:
#@title ⚙️ 준비 — 가장 먼저 실행 (모듈·Day 1~3 도구·스트림 데이터)
import os, sys, json, csv, io, math, time, random, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import clear_output

# Day 1~3 에서 만든 도구 (제공) — Day 4 프레임에는 N(노드) 필드가 추가되었다
class FrameError(Exception):
    pass

def parse_frame(line):
    """$N=..,T=..,H=..,D=..* → (node_id, float, int, int). 검증 실패 시 FrameError."""
    line = line.strip()
    if not (line.startswith("$") and line.endswith("*")):
        raise FrameError(f"손상 프레임: {line!r}")
    d = {}
    for f in line[1:-1].split(","):
        k, v = f.split("=")
        d[k] = v
    return (d["N"], float(d["T"]), int(d["H"]), int(d["D"]))

os.makedirs("data", exist_ok=True); os.makedirs("out", exist_ok=True)
_BOOT = """$N=ESP32-01,T=26.2,H=69,D=220*
$N=ESP32-02,T=32.2,H=70,D=130*
$N=ESP32-03,T=22.4,H=42,D=220*
$N=ESP32-01,T=26.5,H=73,D=103*
$N=ESP32-02,T=30.0,H=41,D=220*
$N=ESP32-03,T=22.8,H=54,D=168*
$N=ESP32-01,T=25.8,H=57,D=220*
$N=ESP32-02,T=38.3,H=54,D=108*
$N=ESP32-03,T=22.4,H=46,D=172*
$N=ESP32-01,T=27.4,H=40,D=106*
$N=ESP32-02,T=40.9,H=65,D=142*
$N=ESP32-03,T=25.1,H=57,D=124*
$N=ESP32-01,T=27.9,H=47,D=108*

$N=ESP32-03,T=24.0,H=48,D=220*
$N=
$N=ESP32-02,T=30.6,H=46,D=113*
$N=ESP32-03,T=22.0,H=61,D=172*
$N=ESP32-0
$N=ESP32-02,T=31.3,H=48,D=146*
$N=ESP32-03,T=25.6,H=55,D=143*
$N=ESP32-01,T=24.8,H=64,D=220*
$N=ESP32-02,T=29.4,H=42,D=129*
$N=ESP32-03,T=25.3,H=71,D=220*
$N=ESP32-01,T=27.9,H=48,D=186*
$N=ESP32-02,T=31.5,H=44,D=117*
$N=ESP32-03,T=22.3,H=66,D=110*
$N=ESP32-0
$N=ESP32-02,T=31.6,H=73,D=160*
$N=ESP32-03,T=23.9,H=62,D=105*"""
_D4 = """$N=ES
$N=ESP32-02,T=29.4,H=70,D=129*
$N=ESP32-03,T=24.3,H=51,D=135*
$N=ESP32-01,T=25.2,H=57,D=121*
$N=ESP32-02,T=32.6,H=53,D=118*
$N=ESP
$N=ESP32-01,T=25.2,H=73,D=201*
$N=ESP32-02,T=30.7,H=53,D=134*
$N=ESP32-03,T=24.0,H=52,D=122*
NESP32-01,T=27.9,H=70,D=112
$N=ESP32-02,T=32.2,H=42,D=220*
$N=ESP32-03,T=23.1,H=52,D=118*
$N=ESP32-01,T=26.5,H=45,D=174*
$N=ESP32-02,T=30.6,H=48,D=129*
$N=ESP32-03,T=22.8,H=70,D=155*
NESP32-01,T=24.5,H=66,D=195
$N=ESP32-02,T=2X.5,H=69,D=220*
$N=ESP32-03,T=22.8,H=70,D=220*
$N=ESP32-01,T=2X.5,H=72,D=128*
$N=ESP32-02,T=32.2,H=42,D=160*
$N=ESP32-03,T=22.1,H=59,D=133*
$N=ESP32-01,T=27.5,H=40,D=143*
$N=ESP32-02,T=29.1,H=71,D=220*
$N=ESP32-03,T=25.2,H=40,D=220*
$N=ESP32-01,T=24.8,H=69,D=122*
$N=ESP32-02,T=32.8,H=70,D=132*
$N=ESP32-03,T=24.3,H=62,D=215*
$N=ESP32-01,T=26.9,H=72,D=187*
$N=ESP32-02,T=30.8,H=51,D=205*
$N=ESP32-03,T=25.0,H=69,D=143*
$N=ESP32-01,T=25.1,H=45,D=220*
$N=ESP32-02,T=31.2,H=57,D=130*
$N=ESP32-03,T=24.4,H=49,D=117*
$N=ESP32-01,T=24.5,H=59,D=109*
$N=ESP32-02,T=31.2,H=65,D=216*
$N=ESP32-03,T=25.8,H=66,D=220*
$N=ESP32-01,T=26.1,H=74,D=150*
$N=ESP32-02,T=31.2,H=50,D=220*
$N=ESP32-03,T=23.7,H=57,D=136*
$N=ESP32-01,T=26.5,H=60,D=114*
$N=ESP32-02,T=29.6,H=65,D=138*
$N=ESP32-03,T=2X.5,H=63,D=220*
$N=ESP32-01,T=26.9,H=53,D=122*
$N=ESP32-02,T=32.9,H=40,D=158*
$N=ESP32-03,T=24.7,H=44,D=121*
$N=ESP32-01,T=24.2,H=67,D=220*
$N=ESP32-02,T=30.6,H=62,D=186*
$N=ESP32-03,T=22.5,H=63,D=187*
$N=ESP32-01,T=24.5,H=57,D=200*
$N=ESP32-02,T=29.4,H=59,D=104*
$N=ESP32-03,T=22.1,H=75,D=127*
$N=ESP32-01,T=24.0,H=46,D=144*
$N=ESP32-02,T=32.0,H=72,D=109*
$N=ESP32-03,T=23.0,H=56,D=203*
$N=ES
$N=ESP32-02,T=31.6,H=65,D=220*
$N=ESP32-03,T=24.0,H=52,D=184*
$N=ESP32-01,T=24.5,H=48,D=142*
$N=
$N=ESP32-03,T=23.8,H=40,D=220*
$N=ESP32-01,T=25.7,H=69,D=219*
$N=ESP32-02,T=32.7,H=52,D=220*
$N=ESP32-03,T=24.4,H=60,D=111*
$N=ESP32-01,T=25.8,H=61,D=132*
$N=ESP32-02,T=31.2,H=59,D=135*
$N=ESP32-03,T=24.5,H=48,D=164*
$N=ESP32-01,T=24.8,H=71,D=190*
$N=ESP32-02,T=30.2,H=46,D=208*
$N=ESP32-03,T=22.6,H=64,D=220*
$N=ESP32-01,T=24.4,H=53,D=155*
$N=ESP32-02,T=32.4,H=45,D=172*
$N=ESP32-03,T=22.5,H=72,D=110*
$N=ESP32-01,T=26.6,H=67,D=157*
$N=ESP32-02,T=30.1,H=68,D=131*
$N=ESP32-03,T=23.4,H=42,D=171*
$N=ESP32-01,T=24.7,H=54,D=124*
$N=ESP32-02,T=30.2,H=51,D=135*
$N=ESP32-03,T=2X.5,H=64,D=138*
$N=ESP32-01,T=26.7,H=54,D=124*
$N=ESP32-02,T=31.0,H=46,D=137*
$N=ESP32-03,T=25.9,H=50,D=131*
$N=ESP32-01,T=24.4,H=68,D=112*
$N=ESP32-02,T=30.0,H=45,D=118*
$N=ESP32-03,T=22.3,H=40,D=168*
$N=ESP32-01,T=26.4,H=67,D=139*
$N=ESP32-02,T=31.4,H=51,D=194*
$N=ESP32-03,T=22.8,H=71,D=220*
$N=ESP32-01,T=26.2,H=55,D=112*
$N=ESP32-02,T=32.4,H=67,D=220*
$N=ESP32-03,T=24.0,H=68,D=134*
$N=ESP32-01,T=25.9,H=64,D=220*
$N=ESP3
$N=ESP32-03,T=24.1,H=51,D=106*
$N=ESP32-01,T=24.7,H=43,D=124*
$N=ESP32-02,T=31.8,H=69,D=205*
$N=ESP32-03,T=24.2,H=48,D=220*
$N=ESP32-01,T=26.5,H=58,D=106*
$N=ESP32-02,T=31.6,H=64,D=140*
$N=ESP32-03,T=24.8,H=43,D=185*

$N=ESP32-02,T=31.2,H=58,D=181*
$N=ESP32-03,T=25.0,H=52,D=114*
$N=ESP32-01,T=24.5,H=74,D=216*
$N=ESP32-02,T=31.9,H=72,D=220*
$N=ESP32-03,T=24.2,H=47,D=108*
$N=ESP32-01,T=24.1,H=74,D=133*
$N=ESP32-02,T=32.1,H=46,D=126*
$N=ESP32-03,T=24.5,H=68,D=109*
$N=ESP32-01,T=26.1,H=58,D=220*
$N=ESP32-02,T=29.9,H=51,D=200*
$N=ESP32-03,T=23.6,H=74,D=143*
$N=ESP32-01,T=24.2,H=58,D=124*
$N=ESP32-02,T=30.1,H=70,D=129*
$N=ESP32-03,T=25.7,H=73,D=197*
$N=ESP32-01,T=27.6,H=58,D=153*
$N=ESP32-02,T=32.8,H=40,D=119*
$N=ESP32-03,T=24.9,H=59,D=219*
$N=ESP32-01,T=25.6,H=48,D=220*
$N=ESP32-02,T=29.9,H=71,D=182*
$N=ESP32-03,T=22.7,H=62,D=146*
$N=ESP32-01,T=25.7,H=56,D=189*
$N=ESP32-02,T=30.1,H=53,D=220*
$N=ESP32-03,T=24.5,H=59,D=127*
$N=ESP32-01,T=25.1,H=43,D=206*
$N=ESP32-02,T=30.5,H=46,D=172*
$N=ESP32-03,T=24.8,H=51,D=107*
$N=ESP32-01,T=25.0,H=54,D=136*
$N=ESP32-02,T=32.5,H=53,D=134*
$N=ESP32-03,T=22.5,H=68,D=220*
$N=ESP32-01,T=25.6,H=48,D=206*
$N=ESP32-02,T=30.6,H=41,D=185*
$N=ESP32-03,T=23.6,H=45,D=137*
$N=ESP32-01,T=25.2,H=56,D=131*
$N=ESP32-02,T=29.4,H=68,D=113*
$N=ESP32-03,T=25.1,H=53,D=206*
$N=ESP32-01,T=24.5,H=43,D=113*
$N=ESP32-02,T=31.7,H=46,D=140*
$N=ESP32-03,T=25.1,H=75,D=124*
$N=ESP32-01,T=24.4,H=49,D=220*
$N=ESP32-02,T=30.5,H=66,D=204*
$N=ESP32-03,T=24.9,H=53,D=201*
$N=ESP32-01,T=25.2,H=57,D=142*
$N=ESP32-02,T=32.3,H=44,D=170*
$N=ESP32-03,T=22.8,H=49,D=140*
$N=ESP32-01,T=27.1,H=74,D=120*
$N=ESP32-02,T=30.8,H=72,D=126*
$N=ESP32-03,T=25.4,H=74,D=220*
$N=ESP32-01,T=27.3,H=48,D=203*
$N=ESP32-02,T=29.4,H=74,D=220*
$N=ESP32-03,T=22.5,H=52,D=189*
$N=ESP32-01,T=24.4,H=44,D=131*
$N=ESP32-02,T=29.5,H=60,D=220*
$N=ESP32-03,T=22.4,H=67,D=118*
$N=ESP32-01,T=24.6,H=46,D=195*
$N=ESP32-02,T=32.6,H=40,D=175*
$N=ESP32-03,T=25.7,H=69,D=145*
$N=ESP32-01,T=26.8,H=57,D=115*
$N=ESP32-02,T=29.6,H=52,D=220*
$N=ESP32-03,T=25.7,H=56,D=122*
$N=ESP3
$N=ESP32-02,T=32.1,H=72,D=158*
$N=ESP32-03,T=23.2,H=40,D=205*
$N=ESP32-01,T=27.7,H=44,D=220*
$N=ESP32-02,T=30.8,H=52,D=122*
$N=ESP32-03,T=24.5,H=72,D=111*
$N=ESP32-01,T=25.9,H=66,D=128*
$N=ESP32-02,T=30.7,H=62,D=176*
$N=ESP32-03,T=24.1,H=45,D=103*
$N=ESP32-01,T=27.5,H=73,D=220*

$N=ESP32-03,T=24.0,H=68,D=142*
$N=ESP32-01,T=26.7,H=68,D=130*
$N=ESP32-02,T=31.8,H=75,D=125*
$N=ESP32-03,T=23.8,H=73,D=112*
$N=ESP32-01,T=25.2,H=56,D=127*
$N=ESP32-02,T=30.3,H=61,D=122*
$N=ESP32-03,T=23.3,H=40,D=220*
$N=ESP32-01,T=25.1,H=67,D=126*
$N=ESP32-02,T=2X.5,H=44,D=118*
$N=ESP32-
$N=ESP32-01,T=25.8,H=52,D=170*
$N=ESP32-02,T=29.7,H=71,D=220*
$N=ESP32-03,T=2X.5,H=50,D=132*
$N=ESP32-01,T=25.1,H=47,D=146*
$N=ESP32-02,T=31.2,H=67,D=220*
$N=ESP32-03,T=24.8,H=71,D=220*
$N=ESP32-01,T=26.0,H=47,D=131*
$N=ESP32-02,T=30.7,H=69,D=220*
$N=ESP32-03,T=22.1,H=45,D=220*
$N=ESP32-01,T=25.6,H=75,D=134*
$N=ESP32-02,T=29.3,H=69,D=123*
$N=ESP32-03,T=25.5,H=53,D=100*
$N=ESP32-01,T=26.4,H=58,D=113*
$N=ESP32-02,T=31.9,H=55,D=220*
$N=ESP32-03,T=24.4,H=47,D=220*
$N=ESP
$N=ESP32-02,T=29.4,H=57,D=129*
$N=ESP32-03,T=25.1,H=54,D=164*
$N=ESP32-01,T=25.8,H=72,D=143*
NESP32-02,T=30.9,H=60,D=220
$N=ESP32-03,T=25.3,H=48,D=191*
$N=ESP32-01,T=25.5,H=42,D=104*
$N=ESP32-02,T=38.3,H=59,D=200*
$N=ESP32-03,T=24.7,H=75,D=220*

$N=ESP32-02,T=37.6,H=60,D=128*
$N=ESP32-03,T=25.4,H=42,D=191*
$N=ESP32-01,T=26.4,H=52,D=151*
$N=ESP32-02,T=40.8,H=53,D=220*
$N=ESP32-03,T=25.8,H=51,D=104*
$N=ESP32-01,T=27.6,H=48,D=158*
$N=ESP32-02,T=39.4,H=66,D=220*

$N=ESP32-01,T=2X.5,H=53,D=220*
$N=ESP32-02,T=40.8,H=55,D=139*
$N=ESP32-03,T=23.1,H=48,D=220*
$N=ESP32-01,T=24.3,H=62,D=135*
$N=ESP32-02,T=39.5,H=74,D=166*
$N=ESP32-03,T=24.0,H=41,D=105*
$N=ESP32-01,T=27.7,H=67,D=210*

$N=ESP32-03,T=24.8,H=58,D=103*
$N=ESP32-01,T=24.7,H=60,D=220*
$N=ESP32-02,T=39.5,H=63,D=120*
$N=ESP32-03,T=22.0,H=58,D=139*
$N=ESP32-01,T=26.9,H=50,D=141*
$N=ESP32-02,T=39.8,H=53,D=220*
$N=ESP32-03,T=24.5,H=50,D=115*
$N=ESP32-01,T=26.6,H=41,D=115*
$N=ESP32-02,T=37.7,H=63,D=133*
$N=ESP32-03,T=22.1,H=49,D=182*
$N=ESP32-01,T=26.5,H=48,D=220*
$N=ESP32-02,T=37.8,H=52,D=124*
$N=ESP32-03,T=22.2,H=66,D=220*
$N=ESP32-01,T=25.9,H=67,D=220*
$N=ESP32-02,T=37.9,H=75,D=218*
$N=ESP32-03,T=24.3,H=75,D=220*
$N=ESP32-01,T=25.5,H=52,D=104*
$N=ESP32-02,T=38.9,H=62,D=132*


$N=ESP32-02,T=40.7,H=68,D=193*

$N=ESP32-01,T=24.8,H=64,D=220*
$N=ESP32-02,T=38.0,H=44,D=121*

NESP32-01,T=25.5,H=42,D=108
$N=ESP32-02,T=38.9,H=61,D=113*
$N=ESP32-03,T=22.4,H=56,D=151*
$N=ESP32-01,T=24.2,H=73,D=131*
$N=ESP32-02,T=32.2,H=58,D=220*
$N=ESP32-03,T=23.4,H=48,D=108*
$N=ESP32-01,T=25.4,H=67,D=148*

$N=ESP32-03,T=25.0,H=64,D=220*
$N=ESP32-01,T=27.2,H=58,D=125*
$N=ESP32-02,T=31.8,H=49,D=203*
$N=ESP32-03,T=23.3,H=44,D=127*
$N=ESP32-01,T=26.0,H=62,D=219*
$N=ESP32-02,T=30.3,H=60,D=175*
$N=ESP32-03,T=25.0,H=51,D=220*
$N=ESP32-01,T=25.9,H=69,D=220*
$N=ESP32-02,T=30.6,H=55,D=183*
$N=ESP32-03,T=23.9,H=60,D=124*
$N=ESP32-01,T=24.3,H=44,D=109*
$N=ESP32-02,T=29.7,H=55,D=126*
$N=ESP32-03,T=24.2,H=63,D=119*
$N=ESP32-01,T=27.3,H=73,D=133*
NESP32-02,T=32.9,H=70,D=129
$N=ESP32-03,T=22.6,H=47,D=109*
$N=ESP32-01,T=26.8,H=46,D=220*

$N=ESP32-03,T=22.4,H=43,D=220*
NESP32-01,T=27.3,H=75,D=220
$N=ESP32-02,T=30.7,H=60,D=122*
$N=ESP32-03,T=25.7,H=44,D=199*
$N=ESP32-01,T=25.9,H=65,D=220*
$N=ESP32-02,T=29.6,H=61,D=107*
$N=ESP32-03,T=23.2,H=54,D=220*
$N=ESP32-01,T=27.0,H=45,D=220*
$N=ESP32-02,T=30.9,H=44,D=115*
$N=ESP32-03,T=24.8,H=43,D=220*
$N=ESP32-01,T=26.0,H=41,D=220*
$N=ESP32-02,T=31.0,H=70,D=123*
$N=ESP32-03,T=22.6,H=43,D=112*
$N=ESP32-01,T=26.3,H=46,D=220*
$N=ESP32-02,T=32.1,H=54,D=220*
$N=ESP32-03,T=24.3,H=61,D=127*
$N=ESP32-01,T=26.6,H=64,D=163*
$N=ESP32-02,T=30.1,H=43,D=220*
$N=ESP32-03,T=25.2,H=56,D=198*
$N=ESP32-01,T=24.3,H=49,D=205*
$N=ESP32-02,T=32.2,H=63,D=220*
NESP32-03,T=23.9,H=46,D=169
$N=ESP32-01,T=27.4,H=74,D=220*
$N=ESP32-02,T=31.4,H=70,D=128*
$N=ESP32-03,T=25.1,H=68,D=220*
$N=ESP32-01,T=25.7,H=50,D=220*
$N=ESP32-02,T=2X.5,H=59,D=192*
$N=ESP32-03,T=24.4,H=44,D=220*"""
open("data/stream_boot.txt", "w", encoding="utf-8").write(_BOOT + "\n")
open("data/stream_day4.txt", "w", encoding="utf-8").write(_D4 + "\n")
BOOT_LINES = _BOOT.split("\n")

RAWS = np.array([512, 723, 1023, 100, 350, 890, 640, 210, 999, 480,
                 305, 760, 150, 555, 830, 420, 970, 60, 700, 260])

_recs = []
for _l in BOOT_LINES:
    if not _l.strip(): continue
    try: _recs.append(parse_frame(_l))
    except (FrameError, ValueError): pass
BT = np.array([r[1] for r in _recs])          # boot 온도 26개
BN = np.array([r[0] for r in _recs])          # boot 노드 26개
RECORDS_BOOT = [{"node": r[0], "T": r[1], "H": r[2], "D": r[3]} for r in _recs]

_y, _x = np.mgrid[0:60, 0:80]
IMG = ((_x + _y) / (79 + 59)).astype(float)   # 흑백 그라디언트 (60, 80)

print("준비 완료 — stream_boot 30줄 · stream_day4 300줄 · BT/BN/RECORDS_BOOT/IMG")

---
# 1교시 · NumPy 기초 — 벡터 연산·브로드캐스팅 (09:00–09:50)

**학습 목표**
- 리스트와 ndarray 의 차이를 설명하고 shape/dtype 을 읽는다
- 반복문 없이 배열 전체를 변환한다 — 벡터 연산·브로드캐스팅

🎬 **D4-1** ① 리스트 vs 배열 → ② 스칼라가 퍼짐 → ③ (2,3)+(3,) → ④ 모양 불일치

### ▶ 예제 1-1 · 어제까지의 방법 vs NumPy — 속도 대결

In [ ]:
raws_list = list(RAWS) * 500                    # 10,000개
%timeit -n 50 [r / 1023 * 3.3 for r in raws_list]   # Day 2 컴프리헨션

big = np.array(raws_list)
%timeit -n 50 big / 1023 * 3.3                       # NumPy 벡터 연산
# 반복문이 C 내부로 들어가면 이만큼 빨라진다 — 8교시 C/C++ 예고의 복선

### ▶ 예제 1-2 · ndarray 만들기 — shape 부터 읽기

In [ ]:
a = np.array([512, 723, 1023])
print(a.shape, a.dtype, a.ndim)      # (3,) int64 1

M = np.arange(12).reshape(3, 4)
print(M.shape, M.ndim)               # (3, 4) 2
print(M)

### ▶ 예제 1-3 · 벡터 연산 — 같은 기호, 다른 의미
🎬 D4-1 ①②

In [ ]:
print([1, 2] * 2)                    # 리스트: 이어 붙이기
print(np.array([1, 2]) * 2)          # 배열: 원소별 곱

volts = np.round(RAWS / 1023 * 3.3, 2)
print(volts[:5])                     # 컴프리헨션 한 줄이 연산 한 줄로

temps = np.array([24.1, 36.9, 28.3, 35.4])
print(temps > 35)                    # 비교도 배열로 — 2교시 마스크의 재료

### ▶ 예제 1-4 · 브로드캐스팅 — 늘어나거나, 터지거나
🎬 D4-1 ③④

In [ ]:
M = np.array([[10, 20, 30], [40, 50, 60]])
v = np.array([1, 2, 3])
print(M + v)                         # (2,3)+(3,) — v 가 행마다 늘어남

try:
    np.array([1, 2, 3]) + np.array([1, 2, 3, 4])
except ValueError as e:
    print("ValueError:", e)          # shapes (3,) (4,) — 메시지에서 모양을 읽자

#### ✏️ 빈칸 채우기 1-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀로 확인합니다.

In [ ]:
# ADC 원시값 → 전압 (round 2) — 반복문 금지
volts = np.round(RAWS ____ 1023 ____ 3.3, 2)
print(volts[:5])         # [1.65 2.33 3.3  0.32 1.13]

In [ ]:
#@title 🔒 정답 보기 1-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAxLTEg4pSA4pSACnZvbHRzID0gbnAucm91bmQoUkFXUyAvIDEwMjMgKiAzLjMsIDIp"
).decode("utf-8"))

#### ✏️ 빈칸 채우기 1-2
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀로 확인합니다.

In [ ]:
# 섭씨 배열 → 화씨 배열 한 줄
c = np.array([0.0, 25.0, 36.9])
f = c ____ 9/5 ____ 32
print(f)                 # [ 32.   77.  98.42]

In [ ]:
#@title 🔒 정답 보기 1-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAxLTIg4pSA4pSACmYgPSBjICogOS81ICsgMzI="
).decode("utf-8"))

#### ✏️ 괄호 넣기 1-3
1. 배열을 받으면 가장 먼저 (　　　) 과 dtype 을 확인한다.
2. 리스트의 `* 2` 는 (　　　)이고, 배열의 `* 2` 는 (　　　)이다.
3. (2,3) + (3,) 이 가능한 이유: 작은 쪽이 (　　　) 모양을 맞추기 때문.
4. (3,) + (4,) 는 (　　　) 예외가 난다.

In [ ]:
#@title 🔒 정답 보기 1-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAxLTMg4pSA4pSACjEuIHNoYXBlCjIuIOydtOyWtCDrtpnsnbTquLAgLyDsm5Dshozrs4Qg"
    "6rOxCjMuIOuKmOyWtOuCmAo0LiBWYWx1ZUVycm9y"
).decode("utf-8"))

---
# 2교시 · NumPy 활용 — 마스크·통계·이미지=배열 (10:00–10:50)

🎬 **D4-1** ⑤ 마스크 생성 → ⑥ 골라내기 → ⑦ 집계

### ▶ 예제 2-1 · ⚠ 슬라이스는 뷰 — 참조 모델의 재림

In [ ]:
a = np.arange(10)
b = a[2:5]           # 리스트라면 복사본, NumPy 는 뷰(참조)!
b[0] = 999
print(a)             # a 도 바뀌었다 — 02 참조 모델 그대로

c = a[2:5].copy()    # 독립 사본이 필요하면 copy
c[0] = -1
print(a[2])          # 999 (안전)

### ▶ 예제 2-2 · 불리언 마스크 — 조건으로 고르기
🎬 D4-1 ⑤⑥

In [ ]:
print(BT[:8])
mask = BT > 35
print(mask[:8])                       # 조건이 데이터가 되었다
print(BT[mask])                       # True 자리만 — ALERT 측정값
print(BT[BN == "ESP32-02"].mean())    # 노드 마스크 + 통계 조합

### ▶ 예제 2-3 · 통계 — 몇 건이, 언제
🎬 D4-1 ⑦

In [ ]:
print((BT > 35).sum())               # 건수 — True 는 1
print(BT.argmax(), BT.max())         # 위치와 값 — "언제 최고였나"

M2 = BT[:24].reshape(4, 6)           # 4일 × 6회라고 가정
print(np.round(M2.mean(axis=1), 2))  # axis=1: 열이 사라짐 → 일별 평균
print(np.round(M2.mean(axis=0), 2))  # axis=0: 행이 사라짐 → 회차별 평균

### ▶ 예제 2-4 · 이미지 = H×W 숫자 배열

In [ ]:
print(IMG.shape, IMG.min(), IMG.max())        # (60, 80) 0.0 1.0
fig, ax = plt.subplots(1, 3, figsize=(10, 2.6))
for a_, im, t in zip(ax, [IMG, IMG[::-1], IMG * 0.5], ["원본", "[::-1] 상하 반전", "* 0.5 어둡게"]):
    a_.imshow(im, cmap="gray", vmin=0, vmax=1); a_.set_title(t); a_.axis("off")
plt.show()
# 다음 주 카메라 프레임이 정확히 이 배열(H, W, 3)입니다

#### ✏️ 빈칸 채우기 2-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀로 확인합니다.

In [ ]:
# 노드별 평균 — 마스크 조합
avg_e2 = BT[BN ____ "ESP32-02"].____()
print(round(float(avg_e2), 2))

In [ ]:
#@title 🔒 정답 보기 2-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTEg4pSA4pSACmF2Z19lMiA9IEJUW0JOID09ICJFU1AzMi0wMiJdLm1lYW4oKQ=="
).decode("utf-8"))

#### ✏️ 괄호 넣기 2-2
1. NumPy 슬라이스 b = a[2:5] 는 복사가 아니라 (　　　)다 — 수정하면 원본도 바뀐다.
2. `(BT > 35).sum()` 이 건수가 되는 이유: True 가 (　　　) 로 계산되기 때문.
3. `argmax` 는 최댓값의 (　　　) 를 돌려준다.
4. axis 는 '(　　　) 방향' — axis=0 이면 행들이 합쳐진다.

In [ ]:
#@title 🔒 정답 보기 2-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAyLTIg4pSA4pSACjEuIOu3sCjssLjsobApCjIuIDEKMy4g7JyE7LmYKOyduOuNseyKpCkK"
    "NC4g7IKs65287KeA64qU"
).decode("utf-8"))

---
# 3교시 · Matplotlib — 시계열·히스토그램 (11:00–11:50)

읽히는 그래프 4요소: **title / xlabel / ylabel / grid** (+범례). 저장은 **savefig 를 show 앞에**.

### ▶ 예제 3-1 · 첫 그래프 — 3줄이면 그림, 4요소면 보고서

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(BT, marker="o", label="T")
plt.title(f"stream_boot temperatures ({len(BT)} readings)")   # 제목에 근거(건수)
plt.xlabel("sample #"); plt.ylabel("Temp (C)")
plt.grid(True); plt.legend()
plt.show()

### ▶ 예제 3-2 · 센서 문법 — 임계선과 ALERT 마킹

In [ ]:
mask = BT > 35
plt.figure(figsize=(8, 3.5))
plt.plot(BT, marker="o", label="T")
plt.axhline(35, color="red", linestyle="--", label="ALERT 35C")
plt.scatter(np.where(mask)[0], BT[mask], color="red", zorder=3, label="ALERT")
plt.title("boot with alerts"); plt.xlabel("#"); plt.ylabel("C"); plt.grid(True); plt.legend()
plt.show()
# 2교시 마스크가 그래프 위의 빨간 점이 되었다

### ▶ 예제 3-3 · 히스토그램 — 분포를 보면 이상이 보인다

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(BT, bins=12)
plt.title("temperature distribution"); plt.xlabel("Temp (C)"); plt.ylabel("count"); plt.grid(True)
plt.show()

### ▶ 예제 3-4 · 저장 — savefig 를 show 앞에

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(BT, bins=12); plt.title("hist"); plt.xlabel("C"); plt.ylabel("n")
plt.savefig("out/demo_hist.png", dpi=150)   # ← 먼저 저장
plt.show()                                  # ← 그 다음 표시 (순서 바꾸면 0바이트!)
print(os.path.getsize("out/demo_hist.png"), "bytes")

#### ✏️ 빈칸 채우기 3-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀로 확인합니다.

In [ ]:
# 시계열 + 임계선 저장
plt.figure(figsize=(8, 3))
plt.plot(BT, marker="o")
plt.____(35, color="red", linestyle="--")
plt.title("boot"); plt.xlabel("#"); plt.ylabel("C"); plt.grid(True)
plt.____("out/blank31.png", dpi=120)
plt.show()
print(os.path.exists("out/blank31.png"))    # True

In [ ]:
#@title 🔒 정답 보기 3-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAzLTEg4pSA4pSACnBsdC5heGhsaW5lKDM1LCBjb2xvcj0icmVkIiwgbGluZXN0eWxlPSIt"
    "LSIpCnBsdC5zYXZlZmlnKCJvdXQvYmxhbmszMS5wbmciLCBkcGk9MTIwKQ=="
).decode("utf-8"))

#### ✏️ 괄호 넣기 3-2
1. 읽히는 그래프의 4요소는 title / xlabel / ylabel / (　　　) 이다.
2. ALERT 기준선은 plt.(　　　)(35, ...) 로 긋는다.
3. png 가 0바이트가 되는 흔한 원인: (　　　) 를 show 뒤에 불렀기 때문.
4. 그래프 제목에 '3 nodes, 264 readings' 처럼 쓰는 이유: 그래프는 주장, 제목은 (　　　).

In [ ]:
#@title 🔒 정답 보기 3-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSAzLTIg4pSA4pSACjEuIGdyaWQo6rKp7J6QKQoyLiBheGhsaW5lCjMuIHNhdmVmaWcKNC4g"
    "6re86rGw"
).decode("utf-8"))

---
# 4교시 · pandas 맛보기 — DataFrame·groupby·CSV (13:00–13:50)

🎬 **D4-2** ① 레코드→표 → ③ 필터 → ⑤ groupby 3단 → ⑥ CSV 왕복

### ▶ 예제 4-1 · DataFrame = dict 레코드 목록의 완성형 — 3종 세트

In [ ]:
df = pd.DataFrame(RECORDS_BOOT)      # Day 2 의 list of dict 가 그대로 표로
print(df.head())
df.info()                            # dtype 확인 — T 가 float 인지!
print(df.describe().round(2))

### ▶ 예제 4-2 · 고르기 — NumPy 마스크와 같은 문법
🎬 D4-2 ②③④

In [ ]:
print(df["T"].mean())                          # 열 선택 → 통계
df["alert"] = df["T"] > 35                     # 판정이 표의 열이 된다
print(df[df["alert"]])                         # 불리언 필터 — 행 걸러내기
print(df.sort_values("T", ascending=False).head(3))

### ▶ 예제 4-3 · groupby — 분할·집계·결합 한 줄
🎬 D4-2 ⑤

In [ ]:
summary = df.groupby("node")["T"].agg(["mean", "max", "count"]).round(2)
print(summary)
# Day 2 for 문 → Day 3 stats_by_node 메서드 → 오늘 한 줄. 3일의 진화 완성.

### ▶ 예제 4-4 · CSV 왕복과 결측
🎬 D4-2 ⑥

In [ ]:
df.to_csv("out/readings_boot.csv", index=False)     # index=False — 유령 열 방지
back = pd.read_csv("out/readings_boot.csv")
print(len(back) == len(df))                          # 왕복 검증 (Day 3 습관)

raw = "node,T\nE1,24.1\nE2,\nE1,28.3"
df2 = pd.read_csv(io.StringIO(raw))
print(df2["T"].isna().sum(), "개 결측 →", len(df2.dropna()), "행")

#### ✏️ 빈칸 채우기 4-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀로 확인합니다.

In [ ]:
# 노드별 요약 한 줄
summary = df.____("node")["T"].____(["mean", "max", "count"]).round(2)
print(summary)

In [ ]:
#@title 🔒 정답 보기 4-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTEg4pSA4pSACnN1bW1hcnkgPSBkZi5ncm91cGJ5KCJub2RlIilbIlQiXS5hZ2coWyJt"
    "ZWFuIiwgIm1heCIsICJjb3VudCJdKS5yb3VuZCgyKQ=="
).decode("utf-8"))

#### ✏️ 빈칸 채우기 4-2
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀로 확인합니다.

In [ ]:
# 저장 → 복원 왕복 — 유령 열 없이
df.to_csv("out/rt.csv", ____=False)
back = pd.____("out/rt.csv")
print(list(back.columns) == list(df.columns))    # True

In [ ]:
#@title 🔒 정답 보기 4-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTIg4pSA4pSACmRmLnRvX2Nzdigib3V0L3J0LmNzdiIsIGluZGV4PUZhbHNlKQpiYWNr"
    "ID0gcGQucmVhZF9jc3YoIm91dC9ydC5jc3YiKQ=="
).decode("utf-8"))

#### ✏️ 괄호 넣기 4-3
1. DataFrame 에서 행은 (　　　), 열은 (　　　)다.
2. `df[df["T"] > 35]` 는 NumPy 의 (　　　) 인덱싱과 같은 문법이다.
3. groupby 의 3단계는 (　　　) → 집계 → 결합.
4. `df["T"] > 35` 가 전부 False 라면 T 의 dtype 이 (　　　)일 가능성 — df.info() 로 확인.

In [ ]:
#@title 🔒 정답 보기 4-3 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA0LTMg4pSA4pSACjEuIOugiOy9lOuTnCAvIO2VhOuTnAoyLiDrtojrpqzslrgg66eI7Iqk"
    "7YGsCjMuIOu2hO2VoAo0LiDrrLjsnpDsl7Qob2JqZWN0KQ=="
).decode("utf-8"))

---
# 5교시 · 파이썬 문법 마무리 — 제너레이터·이터레이터 (14:00–14:50)

🎬 **D3-5** 다시 보기 — "스트림은 본질적으로 제너레이터, readline 이 yield 였던 것"

### ▶ 예제 5-1 · enumerate / zip — 인덱스 순회 졸업

In [ ]:
for i, line in enumerate(BOOT_LINES[:4], 1):
    print(i, repr(line[:24]))

nodes = ["E1", "E2", "E3"]; avgs = [26.1, 32.4, 24.0]
print(dict(zip(nodes, avgs)))               # 쌍 → dict 한 줄
for n, t in zip(nodes, avgs):
    print(f"{n:6s}{t:6.1f}")

### ▶ 예제 5-2 · 이터레이터는 한 번 소모 — Day 2 map 회수

In [ ]:
m = map(str, [1, 2])
print(list(m))       # ['1', '2']
print(list(m))       # [] — 이미 흘러갔다. 스트림과 같은 성질

### ▶ 예제 5-3 · 제너레이터 — yield 로 만드는 수신부

In [ ]:
def read_frames(path):
    """정상 프레임만 (nid, t, h, d) 로 한 건씩 내놓는다."""
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                yield parse_frame(line)      # 여기서 멈췄다가, 다음 요청에 이어서
            except (FrameError, ValueError):
                continue

frames = read_frames("data/stream_boot.txt")
print(type(frames))                          # generator — 호출해도 아직 안 돌았다
print(next(frames))                          # 첫 요청에 첫 건
print(sum(1 for _ in frames))                # 나머지 25 — 리스트 없이 셌다
print(sum(1 for _ in frames))                # 0 — 한 번 소모되면 끝

### ▶ 예제 5-4 · 제너레이터 표현식 + f-string 정렬

In [ ]:
avg = sum(t for _, t, _, _ in read_frames("data/stream_boot.txt")) / 26
print(round(avg, 2))                         # 대괄호 없는 컴프리헨션 — 리스트 없이 집계

print(f"{'node':10s}{'avg_T':>8s}{'count':>7s}")
print(f"{'ESP32-01':10s}{26.433:8.2f}{12:7d}")   # 폭·정렬·소수점 — report 가 반듯해진다

#### ✏️ 빈칸 채우기 5-1
`____` 를 채운 뒤 실행하세요. 정답은 아래 **정답 보기** 셀로 확인합니다.

In [ ]:
# 제너레이터 완성 — 빈 줄 건너뛰고 정상만 yield
def frames_only(path):
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                ____ parse_frame(line)
            except (FrameError, ValueError):
                ____
print(sum(1 for _ in frames_only("data/stream_boot.txt")))    # 26

In [ ]:
#@title 🔒 정답 보기 5-1 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA1LTEg4pSA4pSACnlpZWxkIHBhcnNlX2ZyYW1lKGxpbmUpCi4uLgpjb250aW51ZQ=="
).decode("utf-8"))

#### ✏️ 괄호 넣기 5-2
1. 함수에 (　　　) 가 있으면 호출 시 제너레이터가 된다.
2. yield 는 값을 내놓고 (　　　) 다음 요청에 이어서 실행된다.
3. 제너레이터의 최대 장점: (　　　) 를 만들지 않아 100만 줄도 메모리 걱정이 없다.
4. 한 번 소모한 제너레이터를 다시 돌리면 (　　　).
5. `f"{avg:8.2f}"` 는 폭 8, (　　　) 2자리 실수를 뜻한다.

In [ ]:
#@title 🔒 정답 보기 5-2 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA1LTIg4pSA4pSACjEuIHlpZWxkCjIuIOupiOy3hOuLpOqwgAozLiDrpqzsiqTtirgKNC4g"
    "7JWE66y06rKD64+EIOuCmOyYpOyngCDslYrripTri6QKNS4g7IaM7IiY7KCQ"
).decode("utf-8"))

---
# 6교시 · 미니 프로젝트 ① 필수 (15:00–15:50)

**"센서 데이터 수집·시각화 도구"** — 오전의 부품이 전부 조립됩니다.

| # | 필수 | 오전의 부품 |
|---|---|---|
| 1 | 수신·집계 collect() | 5교시 read_frames + Day 2 예외 |
| 2 | 준실시간 그래프 (50건마다 갱신) | 3교시 + clear_output |
| 3 | DataFrame·groupby 요약 | 4교시 |
| 4 | CSV 저장·왕복 | 4교시 |
| 5 | timeline.png · hist.png | 3교시 |

작성·자동 채점은 **실습 워크시트 6교시**에서. 아래는 준실시간 갱신의 구조 데모입니다.

### ▶ 예제 6-1 · 준실시간 갱신 구조 (축소 데모 — boot 30줄)

In [ ]:
temps = []
for k, (nid, t, h, d) in enumerate(read_frames("data/stream_boot.txt"), 1):
    temps.append(t)
    if k % 10 == 0:                          # 프로젝트에서는 50건마다
        clear_output(wait=True)              # 이전 출력 지우고
        plt.figure(figsize=(8, 3))
        plt.plot(temps[-200:], marker=".")   # 최신 200건 창(window)
        plt.axhline(35, color="red", linestyle="--")
        plt.title(f"live — {k} readings"); plt.grid(True)
        plt.show()
print("수집 종료:", len(temps), "건")

---
# 7교시 · 미니 프로젝트 ② 확장·시연 준비 (16:00–16:50)

확장 E1~E6 중 1개 이상. 막히면 🎬 D4-1(마스크) · D4-2(groupby) · 01(비트 플래그).
🔒 모범 답안(필수·확장)은 8교시 리뷰 때 함께 엽니다.

In [ ]:
# 시연 체크리스트 (3분)
# 1분 — 준실시간 수집 데모 (리허설 1회 완료 필수)
# 1분 — 요약표·그래프: "어느 노드가 언제 문제였나"
# 1분 — 코드 하이라이트 1곳 + 배운 것 1개

In [ ]:
#@title 🔒 정답 보기 7-필수 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA3Le2VhOyImCDilIDilIAKIyDtlYTsiJggMSDCtyDsiJjsi6DCt+ynkeqzhApkZWYgY29s"
    "bGVjdChwYXRoKToKICAgIHJlY29yZHMsIGdvb2QsIGJhZCwgc2tpcCA9IFtdLCAwLCAwLCAwCiAgICBmb3IgbGlu"
    "ZSBpbiBvcGVuKHBhdGgsIGVuY29kaW5nPSJ1dGYtOCIpOgogICAgICAgIGlmIG5vdCBsaW5lLnN0cmlwKCk6CiAg"
    "ICAgICAgICAgIHNraXAgKz0gMTsgY29udGludWUKICAgICAgICB0cnk6CiAgICAgICAgICAgIG5pZCwgdCwgaCwg"
    "ZCA9IHBhcnNlX2ZyYW1lKGxpbmUpCiAgICAgICAgZXhjZXB0IChGcmFtZUVycm9yLCBWYWx1ZUVycm9yKToKICAg"
    "ICAgICAgICAgYmFkICs9IDE7IGNvbnRpbnVlCiAgICAgICAgcmVjb3Jkcy5hcHBlbmQoeyJub2RlIjogbmlkLCAi"
    "VCI6IHQsICJIIjogaCwgIkQiOiBkfSk7IGdvb2QgKz0gMQogICAgcmV0dXJuIHJlY29yZHMsIGdvb2QsIGJhZCwg"
    "c2tpcAoKcmVjb3JkcywgZ29vZCwgYmFkLCBza2lwID0gY29sbGVjdCgiZGF0YS9zdHJlYW1fZGF5NC50eHQiKQpw"
    "cmludChnb29kLCBiYWQsIHNraXApICAgICAgICAgICMgMjY2IDIzIDExCgojIO2VhOyImCAzIMK3IERhdGFGcmFt"
    "ZcK37JqU7JW9CmRmcCA9IHBkLkRhdGFGcmFtZShyZWNvcmRzKQpkZnBbImFsZXJ0Il0gPSBkZnBbIlQiXSA+IDM1"
    "CnN1bW1hcnkzMDAgPSBkZnAuZ3JvdXBieSgibm9kZSIpWyJUIl0uYWdnKFsibWVhbiIsICJtYXgiLCAiY291bnQi"
    "XSkucm91bmQoMikKcHJpbnQoc3VtbWFyeTMwMCkKIyBFU1AzMi0wMSAyNS43NCAyNy43IDg3CiMgRVNQMzItMDIg"
    "MzIuMzQgNDAuOCA4OSAg4oaQIOyXtCDsnbTrsqTtirghCiMgRVNQMzItMDMgMjMuOTkgMjUuOSA5MAoKIyDtlYTs"
    "iJggNCDCtyBDU1Yg7KCA7J6lwrfsmZXrs7UKZGZwLnRvX2Nzdigib3V0L3JlYWRpbmdzLmNzdiIsIGluZGV4PUZh"
    "bHNlKQpzdW1tYXJ5MzAwLnRvX2Nzdigib3V0L3NlbnNvcl9zdW1tYXJ5LmNzdiIpCmJhY2szMDAgPSBwZC5yZWFk"
    "X2Nzdigib3V0L3JlYWRpbmdzLmNzdiIpCmFzc2VydCBsZW4oYmFjazMwMCkgPT0gbGVuKGRmcCkKCiMg7ZWE7IiY"
    "IDUgwrcg7LWc7KKFIOq3uOuemO2UhApwbHQuZmlndXJlKGZpZ3NpemU9KDksIDQpKQpmb3IgbmlkIGluIFsiRVNQ"
    "MzItMDEiLCAiRVNQMzItMDIiLCAiRVNQMzItMDMiXToKICAgIHN1YiA9IGRmcFtkZnBbIm5vZGUiXSA9PSBuaWRd"
    "WyJUIl0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQogICAgcGx0LnBsb3Qoc3ViLCBsYWJlbD1uaWQpCnBsdC5heGhs"
    "aW5lKDM1LCBjb2xvcj0icmVkIiwgbGluZXN0eWxlPSItLSIsIGxhYmVsPSJBTEVSVCAzNUMiKQpwbHQudGl0bGUo"
    "ZiIzIG5vZGVzLCB7bGVuKGRmcCl9IHJlYWRpbmdzIik7IHBsdC54bGFiZWwoInNhbXBsZSAjIChwZXIgbm9kZSki"
    "KQpwbHQueWxhYmVsKCJUZW1wIChDKSIpOyBwbHQuZ3JpZChUcnVlKTsgcGx0LmxlZ2VuZCgpCnBsdC5zYXZlZmln"
    "KCJvdXQvdGltZWxpbmUucG5nIiwgZHBpPTE1MCk7IHBsdC5zaG93KCkKCnBsdC5maWd1cmUoZmlnc2l6ZT0oNywg"
    "My41KSkKcGx0Lmhpc3QoZGZwWyJUIl0sIGJpbnM9MjApCnBsdC50aXRsZSgidGVtcGVyYXR1cmUgZGlzdHJpYnV0"
    "aW9uIik7IHBsdC54bGFiZWwoIlRlbXAgKEMpIik7IHBsdC55bGFiZWwoImNvdW50Iik7IHBsdC5ncmlkKFRydWUp"
    "CnBsdC5zYXZlZmlnKCJvdXQvaGlzdC5wbmciLCBkcGk9MTUwKTsgcGx0LnNob3coKQ=="
).decode("utf-8"))

In [ ]:
#@title 🔒 정답 보기 7-확장 — 이 셀을 실행하면 정답이 출력됩니다
import base64 as _b64
print(_b64.b64decode(
    "4pSA4pSAIOygleuLtSA3Le2ZleyepSDilIDilIAKIyBFMiDCtyDshpDsg4Eg7Jyg7ZiVIOumrO2PrO2KuCDigJQg"
    "7JiI7Jm466W8IOuCmOuIoCDsp5Hqs4QKZXJyX2NvdW50cyA9IENvdW50ZXIoKQpmb3IgbGluZSBpbiBvcGVuKCJk"
    "YXRhL3N0cmVhbV9kYXk0LnR4dCIsIGVuY29kaW5nPSJ1dGYtOCIpOgogICAgaWYgbm90IGxpbmUuc3RyaXAoKToK"
    "ICAgICAgICBjb250aW51ZQogICAgdHJ5OgogICAgICAgIHBhcnNlX2ZyYW1lKGxpbmUpCiAgICBleGNlcHQgKEZy"
    "YW1lRXJyb3IsIFZhbHVlRXJyb3IpIGFzIGU6CiAgICAgICAgZXJyX2NvdW50c1t0eXBlKGUpLl9fbmFtZV9fXSAr"
    "PSAxCnByaW50KGRpY3QoZXJyX2NvdW50cykpICAgICAjIHsnRnJhbWVFcnJvcic6IDE1LCAnVmFsdWVFcnJvcic6"
    "IDh9CgojIEUzIMK3IOydtOuPmSDtj4nqt6Ag4oCUIHBhbmRhcyDtlZwg7KSECnQyID0gZGZwW2RmcFsibm9kZSJd"
    "ID09ICJFU1AzMi0wMiJdWyJUIl0ucmVzZXRfaW5kZXgoZHJvcD1UcnVlKQpyb2xsID0gdDIucm9sbGluZyg1KS5t"
    "ZWFuKCkKcHJpbnQocm91bmQoZmxvYXQocm9sbC5pbG9jWy0xXSksIDIpKSAgICAjIDMxLjM2CnBsdC5maWd1cmUo"
    "Zmlnc2l6ZT0oOCwgMykpCnBsdC5wbG90KHQyLCBhbHBoYT0wLjQsIGxhYmVsPSJyYXciKQpwbHQucGxvdChyb2xs"
    "LCBjb2xvcj0ib3JhbmdlIiwgbGFiZWw9InJvbGxpbmcoNSkiKQpwbHQuYXhobGluZSgzNSwgY29sb3I9InJlZCIs"
    "IGxpbmVzdHlsZT0iLS0iKTsgcGx0LmxlZ2VuZCgpOyBwbHQuZ3JpZChUcnVlKQpwbHQudGl0bGUoIkVTUDMyLTAy"
    "IGhlYXQgZXZlbnQiKTsgcGx0LnNob3coKQoKIyBFNCDtnoztirggwrcg7Je0IOydtOuypO2KuCDqtazqsIQg4oCU"
    "IOuniOyKpO2BrOuhnCDsi5zsnpEv64GdIOywvuq4sApob3RfaWR4ID0gbnAud2hlcmUodDIudmFsdWVzID4gMzUp"
    "WzBdCnByaW50KCLsnbTrsqTtirgg6rWs6rCEKOuFuOuTnCDrgrQg7Iic67KIKToiLCBob3RfaWR4Lm1pbigpLCAi"
    "fiIsIGhvdF9pZHgubWF4KCkp"
).decode("utf-8"))

---
# 8교시 · 팀별 시연·코드 리뷰·정리 (17:00–17:50)

## 오늘의 여섯 문장
1. 배열 연산 = 반복문이 C 내부로 (D4-1) — 그래서 빠르다
2. NumPy 슬라이스는 뷰, 조건은 마스크 — 골라내고(⑥) 세고(⑦) 찾는다(argmax)
3. 그래프는 주장, 제목은 근거 — savefig 는 show 앞에
4. DataFrame = list of dict 의 완성형, df[조건] 은 마스크와 같은 문법 (D4-2)
5. groupby 분할→집계→결합 — stats_by_node 3일의 진화가 한 줄로
6. 스트림은 제너레이터 — yield 로 멈췄다 이어서, 한 번 소모되면 끝 (D3-5)

## C/C++ 과목으로의 다리
- NumPy 가 빨랐던 이유 = 안쪽이 C. 다음 과목에서 그 안쪽으로 내려갑니다.
- 포인터 = Day 2 참조 모델(02)의 C 버전 · 구조체 = Day 3 클래스의 C 버전
- 장비 주간: read_frames 의 `open()` 한 줄이 `serial.Serial()` 로 바뀌는 순간을 라이브로

## 제출
- 팀: out/ 4종 + 코드 → Drive `Day4/제출/팀이름/` · 개인: `Day4_이름.ipynb` (제출 요약 실행)
- 4일간 수고하셨습니다 — 변수 하나에서 저장·시각화되는 시스템까지 왔습니다.